# Find comments by meaning and by words

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericksbr/comment_search/blob/notebook-acts/notebooks/quickstart.ipynb)

This notebook has three acts:

- **Setup:** install the tools, connect Google Drive, and check for a GPU. Run this every session.
- **Act 1: Build your index.** Upload your comment file, check the columns, and build the search index. This is slow, and it needs the GPU. You only do it once per file.
- **Act 2: Search.** Load a saved index and search it. This is quick, and you can come back to it any time.

Comment text stays in this Colab session and in your own Google Drive. Nothing is sent to an outside service.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then run each step in order. Every step says **Done when** so you know it worked.

# Setup (run every session)

## Step 0 — Install the tools

Click play. It takes a minute or two. **Done when:** the cell finishes without a red error.

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # 1) The libraries the search needs, installed normally so their own requirements come along.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "bm25s", "nltk", "sentence-transformers", "gradio", "openpyxl"], check=True)
    # 2) The newest commentsearch from the branch, without the pip cache.
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "--force-reinstall", "--no-deps",
                    "commentsearch @ git+https://github.com/ericksbr/comment_search.git@notebook-acts"], check=True)
print("Install step finished.")


## Step 0b — Connect Google Drive, check the GPU, and load the libraries

Click play. A Google sign-in window may open for Drive; allow it. **Done when:** the cell prints a GPU name, or a warning that there is no GPU.

In [ ]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
import torch

from commentsearch.ingest import load_table, prepare_comments
from commentsearch.columns import (
    detect_text_column, detect_id_column, detect_date_column, detect_group_column, detect_name_column,
    suggest_filter_columns,
)
from commentsearch.index import CommentIndex, STEncoder
from commentsearch.search import Searcher
from commentsearch.checks import run_build_checks

if IN_COLAB:
    from google.colab import drive, files
    drive.mount("/content/drive")

import os
DEFAULT_SAVE = os.environ.get("COMMENTSEARCH_INDEX_DIR", "/content/drive/MyDrive/commentsearch_indexes")
save_box = widgets.Text(value=DEFAULT_SAVE, description="Save index to:", layout=widgets.Layout(width="600px"))
display(save_box)
WIDE = widgets.Layout(width="600px")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print("GPU ready:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. Building an index will work but be slow.")
    print("To fix: Runtime > Change runtime type > T4 GPU, then Runtime > Restart session and run Setup again.")

# Act 1 — Build your index (once per file)

## Step 1 — Upload your comment file

Click play. Then click **Choose Files** and select your CSV, Excel, or Parquet file. **Done when:** you see the number of rows and columns.

In [ ]:
if IN_COLAB:
    print("Click 'Choose Files' and select your comment file.")
    uploaded = files.upload()
    FILE_PATH = list(uploaded.keys())[0]
else:
    FILE_PATH = os.environ["COMMENTS_PATH"]  # local testing only

RAW = load_table(FILE_PATH)
print(f"File: {FILE_PATH}")
print(f"Loaded {len(RAW):,} rows and {len(RAW.columns)} columns.")
RAW.head(3)

## Step 2 — Tell the notebook which columns to use

The notebook guesses each column and shows a sample comment. Change any choice that's wrong.

- **Comment text** (required): the column with the comment words.
- **Comment ID** (required): a unique ID for each comment. Choose "(row number)" if your file has none.
- **Date** (optional): enables the date filter. Leave as "(none)" if there is no date.
- **Duplicate group** (optional): a column that marks comments already known to be duplicates, such as Grouper's `group_id`.
- **Filters** (optional): columns you want to search by, such as commenter name. Hold Ctrl (or Cmd) to pick several.

Then choose where to save the index. The default is a folder in your Google Drive.

**Done when:** the sample comment looks like a real comment, and the choices are what you want.

In [ ]:
COLS = list(RAW.columns)
ID_NONE = "(row number)"
text_guess = detect_text_column(RAW)
id_guess = detect_id_column(RAW)
date_guess = detect_date_column(RAW)
group_guess = detect_group_column(RAW)
name_guess = detect_name_column(RAW)
filter_guess = suggest_filter_columns(
    RAW, exclude={text_guess, id_guess or "", date_guess or "", group_guess or ""}
)

text_dd = widgets.Dropdown(options=COLS, value=text_guess, description="Comment text:", layout=WIDE)
id_dd = widgets.Dropdown(options=[ID_NONE] + COLS, value=id_guess or ID_NONE, description="Comment ID:", layout=WIDE)
date_dd = widgets.Dropdown(options=["(none)"] + COLS, value=date_guess or "(none)", description="Date:", layout=WIDE)
group_dd = widgets.Dropdown(options=["(none)"] + COLS, value=group_guess or "(none)", description="Duplicate group:", layout=WIDE)
name_dd = widgets.Dropdown(options=["(none)"] + COLS, value=name_guess or "(none)", description="Commenter name:", layout=WIDE)
filter_box = widgets.SelectMultiple(options=COLS, value=tuple(filter_guess), description="Filters:", rows=8, layout=WIDE)
sample_out = widgets.Output()

def show_sample(*_):
    with sample_out:
        sample_out.clear_output()
        text = RAW[text_dd.value].astype(str)
        nonempty = text[text.str.strip() != ""]
        print(f"Comment text column: {text_dd.value}")
        print(f"Comment ID column:   {id_dd.value}")
        print("Sample comment:")
        print(nonempty.iloc[0][:400] if len(nonempty) else "(no text found in this column)")

for w in (text_dd, id_dd, date_dd, group_dd):
    w.observe(show_sample, names="value")
show_sample()

display(widgets.VBox([text_dd, id_dd, name_dd, date_dd, group_dd, filter_box, sample_out]))

## Step 3 — Build the index

This is the slow step. It splits each comment into passages, turns the passages into numbers on the GPU, and saves everything to the folder you chose. The progress bar shows how far along it is.

If the folder already has an index for this file, the cell tells you and skips the build. To rebuild, set `REBUILD = True` in the cell and run it again.

After the build, the cell runs validation checks. Each line says **PASS** or **FAIL**. **Done when:** the checks print and the summary says all passed.

In [ ]:
REBUILD = False  # set to True to rebuild an index that already exists

ID_COL = None if id_dd.value == ID_NONE else id_dd.value
DATE_COL = None if date_dd.value == "(none)" else date_dd.value
GROUP_COL = None if group_dd.value == "(none)" else group_dd.value
NAME_COL = None if name_dd.value == "(none)" else name_dd.value
META_COLS = list(filter_box.value)
if NAME_COL and NAME_COL not in META_COLS:
    META_COLS.append(NAME_COL)  # the commenter name is shown in the preview, so keep it
TEXT_COL = text_dd.value
INDEX_DIR = Path(save_box.value) / Path(FILE_PATH).stem

if (INDEX_DIR / "meta.json").exists() and not REBUILD:
    print(f"An index already exists at {INDEX_DIR}.")
    print("Go to Step 4 to load it, or set REBUILD = True and run this cell again.")
else:
    t0 = time.perf_counter()
    raw = RAW.copy()
    if ID_COL is None:
        raw["row_number"] = np.arange(1, len(raw) + 1).astype(str)
        ID_USED = "row_number"
    else:
        ID_USED = ID_COL
    comments = prepare_comments(raw, ID_USED, TEXT_COL, META_COLS, date_col=DATE_COL, group_col=GROUP_COL)
    print(f"Kept {len(comments):,} comments; dropped {comments.attrs['dropped_empty']:,} with no text.")

    encoder = STEncoder("BAAI/bge-small-en-v1.5", device=DEVICE)
    index = CommentIndex.build(comments, encoder, cache_dir=INDEX_DIR, group_col=GROUP_COL, show_progress=True)
    index.meta.update({
        "raw_rows": int(len(RAW)),
        "dropped_empty": int(comments.attrs["dropped_empty"]),
        "source_file": Path(FILE_PATH).name,
        "text_column": TEXT_COL,
        "id_column": ID_USED,
        "name_column": NAME_COL,  # None means the user chose (none)
    })
    INDEX_DIR.mkdir(parents=True, exist_ok=True)
    index.save(INDEX_DIR)
    total = time.perf_counter() - t0

    print(f"\nSaved to {INDEX_DIR}")
    print(f"Comments: {index.meta['n_comments']:,}   Passages: {index.meta['n_passages']:,}")
    if DATE_COL:
        missing = int(comments[DATE_COL].isna().sum())
        if missing == len(comments):
            print("The date column has no usable dates, so the date filter is off.")
        else:
            print(f"Dates that did not parse: {missing:,} of {len(comments):,}")
    print(f"Flagged as very short or 'see attached': {int(comments['boilerplate'].sum()):,}")
    print(f"Duplicate groups: {index.comments['dup_group'].nunique():,}")
    print("\nSeconds per step:", {k: round(v, 2) for k, v in index.timings.items()})
    print(f"Total: {total:.0f} s\n")

    checks = run_build_checks(raw_rows=len(RAW), index=index, searcher=Searcher(index, encoder=encoder))
    print("Checks:")
    for c in checks:
        print("  " + c.line())
    print("\nAll checks passed." if all(c.passed for c in checks)
          else "\nSome checks failed. Look at the FAIL lines before you rely on the results.")

## Step 3b — Download a copy of the search index (optional)

The index is already saved to your Drive folder, so you don't need a copy. A copy on your computer lets you keep the index if Drive is unavailable, or move it to another computer. The file is named `<your file>_search-index.zip`.

**Warning:** downloads of a large index can take a long time, sometimes many minutes, on a slow connection. The size is shown first. Skip this step if you don't need the copy.

**Done when:** the browser asks where to save the file, or the cell prints where the copy was saved.

In [ ]:
import shutil, tempfile

size_mb = sum(p.stat().st_size for p in INDEX_DIR.rglob("*") if p.is_file()) / 1e6
print(f"The search index is about {size_mb:,.0f} MB.")
ZIP_DIR = Path("/content") if IN_COLAB else Path(tempfile.gettempdir())
zip_path = Path(shutil.make_archive(
    str(ZIP_DIR / f"{Path(FILE_PATH).stem}_search-index"), "zip", root_dir=INDEX_DIR
))
print(f"Created {zip_path.name} ({zip_path.stat().st_size / 1e6:,.0f} MB)")
if IN_COLAB:
    files.download(str(zip_path))
    print("Download started. Your browser may ask where to save it.")
else:
    print(f"Saved to {zip_path}")

# Act 2 — Search

## Step 4 — Load a saved index

Use this if you built an index earlier, including in a previous session. If you downloaded a search-index zip in Step 3b and you're starting a new session without your Drive folder, the cell below will ask you to upload it first. You don't need the GPU or your original file for this step. Click play, then pick the index from the list. **Done when:** the cell prints the number of comments and the checks.

In [ ]:
import zipfile
if IN_COLAB:
    print("If you downloaded a search-index zip in Step 3b and this is a new session, click 'Choose Files' and select it.")
    print("If not, press Cancel or skip the upload.")
    uploaded_zips = files.upload()
    for name, data in uploaded_zips.items():
        if not name.lower().endswith(".zip"):
            print(f"Skipped {name}: not a zip file.")
            continue
        Path(name).write_bytes(data)
        index_name = name[: -len(".zip")].replace("_search-index", "")
        target = Path(save_box.value) / index_name
        target.mkdir(parents=True, exist_ok=True)
        zipfile.ZipFile(name).extractall(target)
        print(f"Restored {index_name} into {target}")
else:
    print("Uploading a zip is available in Colab. Locally, the index is already in the save folder.")

SAVE_ROOT = Path(save_box.value)
saved_names = sorted(p.parent.name for p in SAVE_ROOT.glob("*/meta.json")) if SAVE_ROOT.exists() else []
saved_dd = widgets.Dropdown(
    options=saved_names or ["(no saved indexes found)"],
    description="Saved index:",
    layout=WIDE,
)
display(saved_dd)

In [ ]:
if saved_dd.value == "(no saved indexes found)":
    print("No saved index found in", SAVE_ROOT, "- run Step 3 first.")
else:
    LOAD_DIR = SAVE_ROOT / saved_dd.value
    index = CommentIndex.load(LOAD_DIR)
    encoder = STEncoder(index.meta["model"], device=DEVICE)
    searcher = Searcher(index, encoder=encoder)
    print(f"Loaded {index.meta['n_comments']:,} comments from {LOAD_DIR}")
    checks = run_build_checks(
        raw_rows=index.meta.get("raw_rows", index.meta["n_comments"]), index=index, searcher=searcher
    )
    for c in checks:
        print("  " + c.line())
    print("\nAll checks passed." if all(c.passed for c in checks) else "\nSome checks failed.")

## Step 5 — Browse the comments in the index

Type a word to see which comments contain it. Leave the box empty to see the first comments. Use this to confirm a comment you expect is there. **Done when:** the table appears.

In [ ]:
search_box = widgets.Text(placeholder="type a word, or leave empty", description="Contains:", layout=WIDE)
browse_out = widgets.Output()
COMMENT_VIEW_COLS = [c for c in index.comments.columns if c not in ("text", "dup_group", "dup_group_size")]

def show_browse(*_):
    c = index.comments
    q = search_box.value.strip()
    mask = c["text"].str.contains(q, case=False, regex=False) if q else pd.Series(True, index=c.index)
    view = c.loc[mask, COMMENT_VIEW_COLS + ["text"]].head(200).copy()
    view["text"] = view["text"].str.slice(0, 250)
    with browse_out:
        browse_out.clear_output()
        print(f"{int(mask.sum()):,} comments match. Showing up to 200.")
        display(view)

search_box.observe(show_browse, names="value")
display(search_box, browse_out)
show_browse()

## Step 6 — Open the search

Click play. The search opens at a link printed under the cell. It opens in a new tab and works only in this Colab session.

- **Exact words:** every word you type must appear in the comment. Word endings are ignored, so "impact" finds "impacts". Put words in quotes to require them in that order.
- **Fuzzy (by meaning):** finds comments whose sentences are similar in meaning. Move the **Similarity cutoff** slider to make matches stricter (higher) or looser (lower). The blue sentences are the ones that matched.
- **Table:** one row per hit. Use **Table columns** to choose which columns show, and click a column header to sort.
- **Show** (right side): Excerpt (shortest), Passage, or Full comment. Click a row to see it there.

**Done when:** the search opens and typing a word shows hits.


In [ ]:
from commentsearch.app import make_app
import urllib.request

demo = make_app(searcher, title="Comment search")
demo.launch(server_name="0.0.0.0", share=False, prevent_thread_lock=True, debug=False)

# Diagnostic: ask the search server for its own page from inside Colab.
try:
    with urllib.request.urlopen(f"http://127.0.0.1:{demo.server_port}/", timeout=10) as r:
        print("Local check:", r.status)
except Exception as e:
    print("Local check failed:", e)

if IN_COLAB:
    from google.colab.output import eval_js
    from IPython.display import HTML, display

    # Colab's proxy address for the search port. It opens in a new tab and is reachable only from this Colab session.
    url = eval_js(f"google.colab.kernel.proxyPort({demo.server_port})")
    print("Proxy address:", url)
    display(HTML(f'<p><a href="{url}" target="_blank" rel="noopener">Open the comment search in a new tab</a></p>'))
else:
    print(f"Open http://127.0.0.1:{demo.server_port}/ in your browser.")


## Optional — Example searches

Replace the example questions with ones that fit your docket. Each one prints its top results. Reranking is off here to keep it quick.

In [ ]:
EXAMPLE_QUERIES = [
    "economic impacts",
    "wildlife and habitat",
    "legal authority",
]
for q in EXAMPLE_QUERIES:
    frame = searcher.search(q, top_k=5, method="rrf", rerank=False, collapse=True)
    print(f"\n=== {q} ===  ({len(frame)} results)")
    for _, r in frame.iterrows():
        print(f"{r['rank']}. {r['comment_id']}  score {r['score']:.3f}  {r['snippet'][:150]}")